# Dashboard data prep

Builds the aggregated tables that feed the Power BI dashboard, from `data/processed/df_scored.parquet` (already scored by the model in `03_model_training.ipynb`).

In [7]:
import pandas as pd

df = pd.read_parquet("../data/processed/df_scored.parquet")
df["ts"] = pd.to_datetime(df["ts"])

print(df.shape)
df.head()

(446306, 20)


,turbine_id,ts,wind_speed_avg,power_avg,expected_power,power_residual,power_residual_pct,power_avg_moving_6,power_avg_delta,wind_speed_avg_moving_6,wind_speed_avg_delta,gear_oil_temp,gen_bearing_front_temp,gen_bearing_rear_temp,front_bearing_temp,rear_bearing_temp,nacelle_temp,is_downtime,anomaly,anomaly_score
0,WT08,2020-01-01 00:10:00,0.894150,-1.487754,-1.026782,-0.460972,0.448948,-1.716803,0.458099,1.133106,-0.477912,52.509998,38.913334,38.200001,51.930000,51.196667,17.285000,0,1,0.109018
1,WT09,2020-01-01 00:10:00,1.475200,-0.831904,-1.153669,0.321765,-0.278906,-1.012227,0.360646,1.351975,0.246450,47.639999,45.371666,38.521667,48.423332,46.544998,17.996666,0,1,0.096469
2,WT10,2020-01-01 00:10:00,1.865125,-1.097232,-0.977267,-0.119965,0.122756,-0.997823,-0.198818,1.805562,0.119125,47.883331,41.186668,39.033333,48.230000,47.181667,17.455000,0,1,0.124184
3,WT05,2020-01-01 00:10:00,1.834662,-1.584139,-1.162515,-0.421624,0.362683,-2.070336,0.972393,1.748994,0.171337,55.066666,41.143333,39.536667,58.078335,54.790001,17.886667,0,1,0.127311
4,WT06,2020-01-01 00:10:00,2.074462,-2.054331,-1.734176,-0.320155,0.184615,-2.001870,-0.104922,2.063044,0.022837,56.126667,39.250000,38.564999,59.681667,54.023335,16.750000,0,1,0.133681


In [8]:
downtime = df[df["is_downtime"] == 1].copy()
downtime["lost_energy_kwh"] = (downtime["expected_power"] - downtime["power_avg"]).clip(lower=0) * (10 / 60)

anomaly_counts = (
    df.assign(is_anomaly=df["anomaly"] == -1)
    .groupby("turbine_id")["is_anomaly"].sum().rename("anomaly_count").reset_index()
)

lost_energy_by_turbine = (
    downtime.groupby("turbine_id")["lost_energy_kwh"].sum().reset_index()
)

turbine_summary = anomaly_counts.merge(lost_energy_by_turbine, on="turbine_id", how="left")
turbine_summary["lost_energy_kwh"] = turbine_summary["lost_energy_kwh"].fillna(0)
turbine_summary = turbine_summary.sort_values("anomaly_count", ascending=False).reset_index(drop=True)

print("Total lost energy across the fleet (kWh):", round(turbine_summary["lost_energy_kwh"].sum(), 1))
turbine_summary

Total lost energy across the fleet (kWh): 830047.1


,turbine_id,anomaly_count,lost_energy_kwh
0,WT10,4727,61655.955057
1,WT02,4673,146408.016789
2,WT09,3819,95331.925796
3,WT07,3801,114418.838046
4,WT04,3293,98921.926712
5,WT06,3116,74375.375139
6,WT01,2933,100641.135570
7,WT08,2893,76523.903464
8,WT05,2805,61770.047700


In [9]:
anomaly_timeseries = (
    df.set_index("ts")
    .groupby("turbine_id")
    .resample("1h")
    .agg(anomaly_score_avg=("anomaly_score", "mean"), is_downtime=("is_downtime", "max"))
    .reset_index()
)

#hours with no readings at all (data gaps, see 03_model_training.ipynb) produce NaN, drop them
anomaly_timeseries = anomaly_timeseries.dropna(subset=["anomaly_score_avg"])

print(anomaly_timeseries.shape)
anomaly_timeseries.head()

(74533, 4)


,turbine_id,ts,anomaly_score_avg,is_downtime
0,WT01,2020-01-01 00:00:00,0.098182,0.0
1,WT01,2020-01-01 01:00:00,0.092982,0.0
2,WT01,2020-01-01 02:00:00,0.061487,0.0
3,WT01,2020-01-01 03:00:00,0.059233,0.0
4,WT01,2020-01-01 04:00:00,0.050370,0.0


In [10]:
turbine_monthly_summary = (
    df.assign(is_anomaly=df["anomaly"] == -1, month=df["ts"].dt.to_period("M").dt.to_timestamp())
    .groupby(["turbine_id", "month"])["is_anomaly"].sum().rename("anomaly_count").reset_index()
)

print(turbine_monthly_summary.shape)
turbine_monthly_summary.head(12)

(108, 3)


,turbine_id,month,anomaly_count
0,WT01,2020-01-01,307
1,WT01,2020-02-01,485
2,WT01,2020-03-01,316
3,WT01,2020-04-01,176
4,WT01,2020-05-01,122
5,WT01,2020-06-01,332
6,WT01,2020-07-01,78
7,WT01,2020-08-01,148
8,WT01,2020-09-01,87
9,WT01,2020-10-01,132


In [11]:
df_sorted = df.sort_values(["turbine_id", "ts"]).reset_index(drop=True)
is_anom = df_sorted["anomaly"] == -1

df_sorted["persistent_anomaly"] = (
    is_anom
    & is_anom.groupby(df_sorted["turbine_id"]).shift(1).fillna(False)
    & is_anom.groupby(df_sorted["turbine_id"]).shift(2).fillna(False)
)

df = df_sorted
print("Persistent anomalies:", df["persistent_anomaly"].sum(), "/ raw anomalies:", (df["anomaly"] == -1).sum())


Persistent anomalies: 22134 / raw anomalies: 32060


In [12]:
import sys
sys.path.append("../src/etl")
from db_connection import engine

events_df = pd.read_sql(
    "SELECT turbine_id, ts_start, ts_end, status FROM events WHERE status IN ('Stop', 'Warning', 'Communication')",engine,
)

window = pd.Timedelta("72h")
lead_times = []

for _, event in events_df.iterrows():
    anomalies_before = df[
        (df["turbine_id"] == event["turbine_id"])
        & (df["persistent_anomaly"])
        & (df["ts"] >= event["ts_start"] - window)
        & (df["ts"] < event["ts_start"])
    ]
    if not anomalies_before.empty:
        lead_times.append(event["ts_start"] - anomalies_before["ts"].min())
    else:
        lead_times.append(pd.NaT)

events_df["lead_time_hours"] = pd.Series(lead_times).dt.total_seconds() / 3600

n_with_warning = events_df["lead_time_hours"].notna().sum()
print(f"Events with some warning in the 72h window: {n_with_warning} / {len(events_df)}")
print(f"Events with zero warning: {len(events_df) - n_with_warning}")

events_df[["turbine_id", "ts_start", "lead_time_hours"]].head()

Events with some warning in the 72h window: 2116 / 2772
Events with zero warning: 656


,turbine_id,ts_start,lead_time_hours
0,WT01,2020-01-02 09:40:12,2.503333
1,WT01,2020-01-07 11:07:43,26.128611
2,WT01,2020-01-07 11:25:09,26.419167
3,WT01,2020-01-07 11:31:53,26.531389
4,WT01,2020-01-07 11:53:51,26.897500


## Export for Power BI

Two clean, aggregated CSVs, ready to import directly. CSV instead of parquet here since Power BI's native CSV import is simpler to set up than the parquet connector, and these tables are small (no performance reason to prefer parquet at this size).

In [13]:
turbine_summary.to_csv("../data/processed/turbine_summary.csv", index=False)
anomaly_timeseries.to_csv("../data/processed/anomaly_timeseries.csv", index=False)
turbine_monthly_summary.to_csv("../data/processed/turbine_monthly_summary.csv", index=False)
events_df[["turbine_id", "ts_start", "status", "lead_time_hours"]].to_csv(
    "../data/processed/lead_times.csv", index=False
)

print("Exported turbine_summary.csv:", turbine_summary.shape)
print("Exported anomaly_timeseries.csv:", anomaly_timeseries.shape)
print("Exported turbine_monthly_summary.csv:", turbine_monthly_summary.shape)
print("Exported lead_times.csv:", events_df.shape)


Exported turbine_summary.csv: (9, 3)
Exported anomaly_timeseries.csv: (74533, 4)
Exported turbine_monthly_summary.csv: (108, 3)
Exported lead_times.csv: (2772, 5)
